#***Capstone Project: Cease & Desist Document Processing***

Built with
- LangChain & LangGraph




**Setup and Installations**



In [28]:
!pip install -qU langchain-groq langgraph langchain-community pypdf langsmith

In [29]:
!apt-get install -qq poppler-utils
!pip install -q pdf2image
print("✅ pdf2image installed successfully!")

✅ pdf2image installed successfully!


In [59]:
import os
from google.colab import userdata

# -- Leverage keys from secrets --
os.environ["GROQ_API_KEY"]= userdata.get('GROQ_API_KEY')
os.environ["LANGCHAIN_API_KEY"] = userdata.get('LANGCHAIN_API_KEY')
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "Cease-desist-agent"

print("Environment setup configured")


Environment setup configured


### Dry run to test the connection

In [57]:
from langchain_groq import ChatGroq

#llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)



response = llm.invoke("""What happened in the indian stock market yesterday.
            Give the information in not more than 2 sentences""")

print(f"LLM Response: {response.content}",end='\n')

print("Connection Established Successfully")

LLM Response: I'm not aware of the current date or the specific events that occurred in the Indian stock market yesterday. However, I can suggest some possible sources where you can find the latest information on the Indian stock market, such as:

- BSE (Bombay Stock Exchange) website
- NSE (National Stock Exchange) website
- Financial news websites like Bloomberg, CNBC, or Economic Times

Please note that my knowledge cutoff is December 2023, and I may not have the most up-to-date information.
Connection Established Successfully


Access sample documents from Google Drive

In [32]:
# from google.colab import drive
# drive.mount('/content/drive')

# #pointing to folder containing the pdfs

# pdf_Folder = "/content/drive/MyDrive/GenAi_Capstone/sample_Docs"

# pdf_Files = sorted(os.listdir(pdf_Folder))

# print(f"Total documents loaded: {len(pdf_Files)}\n")

# for f in pdf_Files:
#   print(f"- {f}")


### Access sample data from google colab folder

In [33]:
pdf_Folder = "/content/CapstoneInputData"

pdf_Files = sorted(os.listdir(pdf_Folder))

for f in pdf_Files:
  print(f"- {f}")

- 01_copyright_infringement_photography.pdf
- 02_trademark_infringement_tech.pdf
- 03_trade_secret_misappropriation.pdf
- 04_defamation_online_review.pdf
- 05_patent_infringement_medical_device.pdf
- 06_harassment_workplace.pdf
- 07_software_license_violation.pdf
- 08_non_compete_violation.pdf
- 09_copyright_infringement_music.pdf
- 10_breach_of_contract_nda.pdf
- LOA2.pdf
- LOA3.pdf
- LOA4.pdf
- LOA5.pdf
- LOA6.pdf
- LOA7.pdf
- LOA8.pdf
- LOA9.pdf
- LoA1.pdf
- bw_doc_1.pdf
- bw_doc_2.pdf
- bw_doc_3.pdf
- bw_doc_4.pdf
- bw_doc_5.pdf
- notice_1.pdf
- notice_2.pdf
- notice_3.pdf
- notice_4.pdf
- notice_5.pdf


### Read PDF Files

- Read each pdf file
- If text, return the full text
- Else using convert_from_path we are splitting the pdfs into images. (one image per page)
- using base64, we are converting the binary images to text which we will be sending via API to groq vision.
- Groq vision reads the image and extracts the text.

In [60]:
from langchain_community.document_loaders import PyPDFLoader
from pdf2image import convert_from_path
import base64
import os
def load_pdf(filename:str) -> str:
  """
  Takes a pdf file, reads from input folder and returns the content as string
  Ff the pdf file scanned image,go to groq vision after trying to read via pypdfloader
  """

  filePath = f"{pdf_Folder}/{filename}"
  loader = PyPDFLoader(filePath)
  pages = loader.load()

  full_text = "\n".join([page.page_content for page in pages])

  if full_text.strip():
    print(f" Pdf file extracted")
    return full_text

  else:
    images = convert_from_path(filePath)
    extracted_text = []

    for i, image in enumerate(images):
      temp_path = f"/content/temp_page_{i}.jpg"
      image.save(temp_path, "JPEG")

      with open(temp_path, "rb") as img_file:
        image_data = base64.b64encode(img_file.read()).decode("utf-8")

      from langchain_groq import ChatGroq
      from langchain_core.messages import HumanMessage

      vision_llm = ChatGroq(model="meta-llama/llama-4-scout-17b-16e-instruct")

      message = HumanMessage(
              content=[
                  {
                      "type": "image_url",
                      "image_url": {
                          "url": f"data:image/jpeg;base64,{image_data}"
                      }
                  },
                  {
                      "type": "text",
                      "text": "Extract all the text from this document image. Return only the extracted text, nothing else."
                  }
              ]
          )
      response = vision_llm.invoke([message])
      extracted_text.append(response.content)





      os.remove(temp_path)

  return "\n".join(extracted_text)
print("PDF files loaded")


PDF files loaded


Testing with sample files

In [35]:
test_files = ["LoA1.pdf", "notice_1.pdf", "bw_doc_1.pdf"]

for singleFile in test_files:
  text = load_pdf(singleFile)
  print(f"File: {singleFile}")
  print(f"Length : {len(text)}\n")



 Pdf file extracted
File: LoA1.pdf
Length : 1680

File: notice_1.pdf
Length : 1470

File: bw_doc_1.pdf
Length : 1321



#CLASSIFICATION AGENT

- Receives extracted text from the PDF
- Analyze and determine the type of request using System prompt
- Return a structured output with fields such as Classification, Confidence, Reason

Defining Output Schema

- We are defining the output schema such that it has to return only 3 fields

1. Document classification
2. Confidence level
3. Reasoning  behind the classification

In [36]:
from pydantic import BaseModel, Field, field_validator
from typing import Literal

class ClassificationResult(BaseModel):
    """
    Defines the exact structure of the LLM's classification response.
    The LLM must always return these 3 fields — nothing more, nothing less.
    """

    classification: Literal["Cease", "Irrelevant", "Uncertain"] = Field(
        description="The document classification result"
    )

    confidence: float = Field(
        description="Confidence score between 0.0 and 1.0",
        ge=0.0,
        le=1.0
    )

    reason: str = Field(
        description="Brief explanation of why this classification was chosen"
    )

    # ── Auto-convert string to float if LLM returns "0.95" ────────
    @field_validator("confidence", mode="before")
    @classmethod
    def parse_confidence(cls, v):
        """
        Converts confidence to float even if LLM returns it as a string.
        Handles cases like "0.95", "95%", "95"
        """
        if isinstance(v, str):
            # Remove % sign if present e.g. "95%" → "95"
            v = v.replace("%", "").strip()
            v = float(v)
            # If value looks like percentage e.g. 95 → convert to 0.95
            if v > 1.0:
                v = v / 100
        return v

print(" Output Schema defined successfully!")
print(f"\nSchema fields:")
print(f"  → classification : Cease / Irrelevant / Uncertain")
print(f"  → confidence     : 0.0 to 1.0 (auto-converts strings)")
print(f"  → reason         : explanation text")

 Output Schema defined successfully!

Schema fields:
  → classification : Cease / Irrelevant / Uncertain
  → confidence     : 0.0 to 1.0 (auto-converts strings)
  → reason         : explanation text


#System Prompt to classify the document

In [48]:
from langchain_core.messages import SystemMessage, HumanMessage


CLASSIFICATION_SYSTEM_PROMPT = """You are a legal document
classification specialist with expertise in Cease & Desist requests.

Your job is to analyze documents and classify them into exactly
one of these 3 categories:

1. Cease — Use this when the document:
   - Explicitly requests to stop all communications
   - Contains legal language demanding cessation
   - Has clear sender and recipient details
   - Mentions legal consequences for non-compliance
   - Uses phrases like cease and desist, stop all contact,
     no further communication

2. Irrelevant — Use this when the document:
   - Has nothing to do with stopping communication
   - Is a general business document, invoice, or letter
   - Does not contain any legal cease & desist language

3. Uncertain — Use this when:
   - The document is unclear or ambiguous
   - You cannot determine the intent with confidence
   - The document is damaged, incomplete or unreadable
   - Confidence score is below 0.75

Confidence scoring guide:
   - 0.90 to 1.00 → Very clear, no doubt
   - 0.75 to 0.89 → Fairly confident
   - Below 0.75   → Use Uncertain classification

Always be precise and conservative. When in doubt use Uncertain.
Ensure the confidence score is returned as a floating-point number (e.g., 0.95).
"""

# Classification function

def classify_document(filename: str, document_text: str) -> ClassificationResult:
    """
    Classifies a document as Cease, Irrelevant or Uncertain.
    Takes filename and text, returns a ClassificationResult object.
    """

    # Step 1: Build the messages list
    messages = [
        SystemMessage(content=CLASSIFICATION_SYSTEM_PROMPT),
        HumanMessage(content=f"""Please classify this document:

Document Name: {filename}
Document Content: {document_text}
""")
    ]

    # Step 2: Connect LLM to our schema and invoke
    structured_llm = llm.with_structured_output(ClassificationResult)
    result = structured_llm.invoke(messages)

    return result

print("Document classification done.")

Document classification done.


##Testing classification with sample files

In [38]:

test_files = ["LoA1.pdf", "notice_1.pdf", "bw_doc_1.pdf"]

for filename in test_files:
    print(f" Processing: {filename}")
    print("-----------------------------------------------------------")

    # Step 1: Load the PDF text
    document_text = load_pdf(filename)

    # Step 2: Classify it
    result = classify_document(filename, document_text)

    # Step 3: Display the result
    print(f"Classification : {result.classification}")
    print(f"Confidence     : {result.confidence:.0%}")
    print(f"Reason         : {result.reason}\n\n")


 Processing: LoA1.pdf
-----------------------------------------------------------
 Pdf file extracted
Classification : Cease
Confidence     : 95%
Reason         : The document explicitly contains the phrase 'Cease and desist all communications regarding this account' and demands that all communications be directed exclusively to the client's agent, PINNACLE LAW GROUP.


 Processing: notice_1.pdf
-----------------------------------------------------------
Classification : Cease
Confidence     : 92%
Reason         : The document contains legal language demanding cessation of specific outreach modes, requests a stop to certain communications, and mentions potential legal consequences for non-compliance, which aligns with the criteria for a Cease classification.


 Processing: bw_doc_1.pdf
-----------------------------------------------------------
Classification : Irrelevant
Confidence     : 95%
Reason         : The document is a notice regarding a limited Power of Attorney representation

## State Definition

In [39]:
from typing import TypedDict, Optional

class DocumentState(TypedDict):
    """
    The shared state that flows through the entire workflow.
    Every agent reads from and writes to this state.
    Fields start as None and get filled as the document
    moves through each node.
    """
    # input
    filename        : str             # PDF filename e.g. "LoA1.pdf"
    document_text   : str             # extracted text from PDF

    # classification fields
    classification  : Optional[str]   # Cease / Irrelevant / Uncertain
    confidence      : Optional[float] # 0.0 to 1.0
    reason          : Optional[str]   # why this classification

    # processing fields
    db_saved        : Optional[bool]  # True if saved to database
    archived        : Optional[bool]  # True if archived to file
    human_reviewed  : Optional[bool]  # True if human reviewed

    # audit fields
    audit_log       : Optional[str]   # full audit trail message
    processed_at    : Optional[str]   # timestamp of processing

print("DocumentState defined ")
print("\nState fields:")
print("  1. Input         : filename, document_text")
print("  2. Classification: classification, confidence, reason")
print("  3. Processing    : db_saved, archived, human_reviewed")
print("  4. Audit         : audit_log, processed_at")

DocumentState defined 

State fields:
  1. Input         : filename, document_text
  2. Classification: classification, confidence, reason
  3. Processing    : db_saved, archived, human_reviewed
  4. Audit         : audit_log, processed_at


## Load and Classify

In [40]:
from datetime import datetime

# Node 1 - Load the pdf input
def load_document_node(state: DocumentState) -> dict:
    """
    Reads the PDF file and extracts its text.
    Updates: document_text
    """
    print(f"Loading document: {state['filename']}")

    text = load_pdf(state["filename"]) # already defined earlier, callign now

    print(f" Extracted {len(text)} characters")


    return {"document_text": text}


# Document classification in this function

def classify_node(state: DocumentState) -> dict:
    """
    Classifies the document as Cease, Irrelevant or Uncertain.
    Updates: classification, confidence, reason
    """
    print(f"Classifying: {state['filename']}")

    result = classify_document(
        filename=state["filename"],
        document_text=state["document_text"]
    )

    print(f" Classification : {result.classification}")
    print(f" Confidence     : {result.confidence:.0%}")

    # Return only the fields we updated
    return {
        "classification" : result.classification,
        "confidence"     : result.confidence,
        "reason"         : result.reason
    }



## Create DB to store the valid cease/desist, Archive irrelevants and Audits

In [41]:
import sqlite3
from datetime import datetime

import logging

# Logger setup
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[
        logging.FileHandler("/content/audit.log"),
        logging.StreamHandler()
    ]
)

logger = logging.getLogger("cease_desist_agent")

# Create Db and archive path
DB_PATH      = "/content/cease_desist.db"
ARCHIVE_PATH = "/content/archive.txt"

def setup_database():
    """
    Creates the SQLite database and table if they don't exist.
    Called once at the start.
    """
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS cease_desist_requests (
            id            INTEGER PRIMARY KEY AUTOINCREMENT,
            filename      TEXT NOT NULL,
            classification TEXT NOT NULL,
            confidence    REAL NOT NULL,
            reason        TEXT NOT NULL,
            processed_at  TEXT NOT NULL
        )
    """)

    conn.commit()
    conn.close()
    print("Database created")

# setup
setup_database()
# DB node
def database_node(state: DocumentState) -> dict:
    """
    Saves valid Cease & Desist requests to SQLite database.
    Updates: db_saved, processed_at
    """
    processed_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    try:
        conn   = sqlite3.connect(DB_PATH)
        cursor = conn.cursor()

        cursor.execute("""
            INSERT INTO cease_desist_requests
            (filename, classification, confidence, reason, processed_at)
            VALUES (?, ?, ?, ?, ?)
        """, (
            state["filename"],
            state["classification"],
            state["confidence"],
            state["reason"],
            processed_at
        ))

        conn.commit()
        conn.close()

        logger.info(
            f"CEASE SAVED TO DB | "
            f"File: {state['filename']} | "
            f"Confidence: {state['confidence']:.0%}"
        )

        return {
            "db_saved"     : True,
            "processed_at" : processed_at
        }

    except Exception as e:
        logger.error(
            f"DB SAVE FAILED | "
            f"File: {state['filename']} | "
            f"Error: {str(e)}"
        )
        return {
            "db_saved"     : False,
            "processed_at" : processed_at
        }


# Archive node
def archive_node(state: DocumentState) -> dict:
    """
    Writes Irrelevant documents to a flat text archive file.
    Updates: archived, processed_at
    """
    processed_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    try:
        with open(ARCHIVE_PATH, "a") as f:
            f.write(
                f"{processed_at} | "
                f"{state['filename']} | "
                f"{state['classification']} | "
                f"{state['reason']}\n"
            )

        logger.info(
            f"IRRELEVANT ARCHIVED | "
            f"File: {state['filename']} | "
            f"Confidence: {state['confidence']:.0%}"
        )

        return {
            "archived"     : True,
            "processed_at" : processed_at
        }

    except Exception as e:
        logger.error(
            f"ARCHIVE FAILED | "
            f"File: {state['filename']} | "
            f"Error: {str(e)}"
        )
        return {
            "archived"     : False,
            "processed_at" : processed_at
        }

# Audit node.
def audit_node(state: DocumentState) -> dict:
    """
    Logs every decision using Python's logging library.
    Updates: audit_log
    """

    # Choose log level based on classification
    if state["classification"] == "Cease":
        logger.info(
            f"CEASE REQUEST | "
            f"File: {state['filename']} | "
            f"Confidence: {state['confidence']:.0%} | "
            f"Reason: {state['reason']} | "
            f"DB Saved: {state.get('db_saved', False)}"
        )
    elif state["classification"] == "Irrelevant":
        logger.info(
            f"IRRELEVANT | "
            f"File: {state['filename']} | "
            f"Confidence: {state['confidence']:.0%} | "
            f"Archived: {state.get('archived', False)}"
        )
    elif state["classification"] == "Uncertain":
        logger.warning(
            f"UNCERTAIN - HUMAN REVIEW NEEDED | "
            f"File: {state['filename']} | "
            f"Confidence: {state['confidence']:.0%} | "
            f"Reason: {state['reason']}"
        )

    audit_message = (
        f"{state['classification']} | "
        f"{state['filename']} | "
        f"Confidence: {state['confidence']:.0%}"
    )

    return {"audit_log": audit_message}

print("\n Database, Archive and Audit nodes defined!")

Database created

 Database, Archive and Audit nodes defined!


## Logic for Human In The Loop

In [42]:
from langgraph.types import interrupt

def hitl_node(state: DocumentState) -> dict:
    """
    Pauses the workflow for human review when classification
    is Uncertain. Presents document details and waits for
    human decision before routing to database or archive.
    Updates: classification, human_reviewed
    """

    # Provide details to Human


    print("Manual Review Required")
    print("--"*20)
    print(f" File name       : {state['filename']}")
    print(f" Confidence score : {state['confidence']:.0%}")
    print(f" Reason     : {state['reason']}")
    print(f"\n Document Preview:")
    print(f"{state['document_text'][:500]}...")

    # Interrupt and wait for human input

    human_decision = interrupt({
        "message"   : "Please review this document and make a decision",
        "filename"  : state["filename"],
        "reason"    : state["reason"],
        "confidence": state["confidence"],
        "options"   : "Type 'cease' or 'irrelevant'"
    })

    # Process Human Decision

    decision = human_decision.strip().lower()

    if decision == "cease":
        logger.info(
            f"HUMAN REVIEW | APPROVED AS CEASE | "
            f"File: {state['filename']}"
        )
        return {
            "classification" : "Cease",
            "human_reviewed" : True
        }

    elif decision == "irrelevant":
        logger.info(
            f"HUMAN REVIEW | REJECTED AS IRRELEVANT | "
            f"File: {state['filename']}"
        )
        return {
            "classification" : "Irrelevant",
            "human_reviewed" : True
        }

    else:
        # If human types something unexpected, flag as uncertain again
        logger.warning(
            f"HUMAN REVIEW | INVALID INPUT: {human_decision} | "
            f"File: {state['filename']} | "
            f"Keeping as Uncertain"
        )
        return {
            "classification" : "Uncertain",
            "human_reviewed" : False
        }

print("HITL node defined!")

HITL node defined!


### Routing Logic

In [43]:
def route_document(state: DocumentState) -> str:
    """
    Reads the classification from state and returns
    the name of the next node to execute.
    This function is used as a conditional edge in the graph.
    """
    classification = state["classification"]

    if classification == "Cease":
        print(f"  🔀 Routing to → Database Node")
        return "database_node"

    elif classification == "Irrelevant":
        print(f"  🔀 Routing to → Archive Node")
        return "archive_node"

    elif classification == "Uncertain":
        print(f"  🔀 Routing to → HITL Node")
        return "hitl_node"

print("Routing logic definition completed")

Routing logic definition completed


### Build Graph

In [44]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

# Create empty graph
graph_builder = StateGraph(DocumentState)

# Add nodes to graph builder
graph_builder.add_node("load_document_node", load_document_node)
graph_builder.add_node("classify_node",      classify_node)
graph_builder.add_node("database_node",      database_node)
graph_builder.add_node("archive_node",       archive_node)
graph_builder.add_node("hitl_node",          hitl_node)
graph_builder.add_node("audit_node",         audit_node)

# Add edges

# Fixed edges — always go to next node
graph_builder.add_edge(START,                "load_document_node")
graph_builder.add_edge("load_document_node", "classify_node")

# Conditional edge — routes based on classification
graph_builder.add_conditional_edges(
    "classify_node",   # from this node
    route_document,    # use this routing function
    {                  # map returned string to actual node
        "database_node" : "database_node",
        "archive_node"  : "archive_node",
        "hitl_node"     : "hitl_node"
    }
)

# After HITL human decides — re-route based on updated classification
graph_builder.add_conditional_edges(
    "hitl_node",       # from this node
    route_document,    # reuse same routing function
    {
        "database_node" : "database_node",
        "archive_node"  : "archive_node",
        "hitl_node"     : "hitl_node"   # if invalid input, retry
    }
)

# After database or archive — always go to audit
graph_builder.add_edge("database_node", "audit_node")
graph_builder.add_edge("archive_node",  "audit_node")

# After audit — always end
graph_builder.add_edge("audit_node", END)

# Compile with memory check pointer
memory = MemorySaver()
graph  = graph_builder.compile(
    checkpointer=memory,
    interrupt_before=["hitl_node"]  # pause BEFORE hitl runs
)

print(" Graph compiled successfully!")
print("\nGraph nodes:")
for node in graph.get_graph().nodes:
    print(f"  → {node}")

 Graph compiled successfully!

Graph nodes:
  → __start__
  → load_document_node
  → classify_node
  → database_node
  → archive_node
  → hitl_node
  → audit_node
  → __end__


# PROCESS ALL DOCUMENTS



In [53]:
from langgraph.types import Command

def process_document(filename: str) -> dict:
    """
    Runs a single document through the complete graph.
    If HITL is triggered, pauses and asks for human input.
    Returns the final state after all nodes have executed.
    """
    thread_id = filename.replace(".pdf", "").replace(" ", "_")
    config    = {"configurable": {"thread_id": thread_id}}

    initial_state = {
        "filename"       : filename,
        "document_text"  : "",
        "classification" : None,
        "confidence"     : None,
        "reason"         : None,
        "db_saved"       : None,
        "archived"       : None,
        "human_reviewed" : None,
        "audit_log"      : None,
        "processed_at"   : None
    }

    print(f"\n{'='*60}")
    print(f"📄 Processing: {filename}")
    print(f"{'='*60}")

    # ── Step 1: Run graph until completion or HITL pause ──────────
    for event in graph.stream(initial_state, config=config):
        pass

    # ── Step 2: Check if workflow paused for HITL ─────────────────
    current_state = graph.get_state(config)

    while current_state.next == ("hitl_node",):

        # Show document details to human reviewer
        print("\n" + "="*60)
        print(" HUMAN REVIEW REQUIRED")
        print("="*60)
        print(f" File       : {current_state.values['filename']}")
        print(f" Confidence : {current_state.values['confidence']:.0%}")
        print(f" Reason     : {current_state.values['reason']}")
        print(f"\n Document Preview:")
        print(f"{current_state.values['document_text'][:500]}...")
        print("="*60)

        # Ask human for decision
        while True:
            human_decision = input(
                "\n Your decision (type 'cease' or 'irrelevant'): "
            ).strip().lower()

            if human_decision in ["cease", "irrelevant"]:
                break
            else:
                print(" Invalid! Please type 'cease' or 'irrelevant'")

        print(f"\nDecision received: {human_decision.upper()}")
        print("▶️  Resuming workflow...\n")

        # Resume graph with human decision
        for event in graph.stream(
            Command(resume=human_decision),
            config=config
        ):
            pass

        # Check if paused again (invalid input loop)
        current_state = graph.get_state(config)

    # ── Step 3: Return final state ────────────────────────────────
    final_state = graph.get_state(config)
    return final_state.values

print("Document processor function ready!")

✅ Document processor function ready!


In [58]:
# Get all PDF files from the folder
all_pdfs = sorted([
    f for f in os.listdir(pdf_Folder)
    if f.endswith(".pdf")
])

print(f"📂 Total documents to process: {len(all_pdfs)}\n")

# Store results for summary
results = []

for filename in all_pdfs:
    result = process_document(filename)
    results.append({
        "filename": filename,
        "result"  : result
    })

print(f"\n✅ All {len(all_pdfs)} documents processed!")

📂 Total documents to process: 29


📄 Processing: 01_copyright_infringement_photography.pdf
Loading document: 01_copyright_infringement_photography.pdf
 Pdf file extracted
 Extracted 3153 characters
Classifying: 01_copyright_infringement_photography.pdf
 Classification : Cease
 Confidence     : 99%
  🔀 Routing to → Database Node

📄 Processing: 02_trademark_infringement_tech.pdf
Loading document: 02_trademark_infringement_tech.pdf
 Pdf file extracted
 Extracted 3246 characters
Classifying: 02_trademark_infringement_tech.pdf
 Classification : Cease
 Confidence     : 99%
  🔀 Routing to → Database Node

📄 Processing: 03_trade_secret_misappropriation.pdf
Loading document: 03_trade_secret_misappropriation.pdf
 Pdf file extracted
 Extracted 3527 characters
Classifying: 03_trade_secret_misappropriation.pdf
 Classification : Cease
 Confidence     : 99%
  🔀 Routing to → Database Node

📄 Processing: 04_defamation_online_review.pdf
Loading document: 04_defamation_online_review.pdf
 Pdf file extract

In [63]:
print("\n" + "="*60)
print("📊 FINAL PROCESSING SUMMARY")
print("="*60)

cease_count      = 0
irrelevant_count = 0
uncertain_count  = 0

for item in results:
    filename       = item["filename"]
    result         = item["result"]
    classification = result.get("classification", "Unknown")
    human_reviewed = result.get("human_reviewed", False)

    if classification == "Cease":
        cease_count += 1
        status = "Cease"
    elif classification == "Irrelevant":
        irrelevant_count += 1
        status = "Irrelevant"
    else:
        uncertain_count += 1
        status = "Uncertain"

    reviewed = "*Human Reviewed*" if human_reviewed else ""
    print(f"{status} | {filename} | {reviewed}")

print("="*60)
print(f"\nTOTALS:")
print(f"  Cease      : {cease_count}")
print(f"  Irrelevant : {irrelevant_count}")
print(f"  Uncertain  : {uncertain_count}")
print(f"  Total      : {len(results)}")



📊 FINAL PROCESSING SUMMARY
Cease | 01_copyright_infringement_photography.pdf | 
Cease | 02_trademark_infringement_tech.pdf | 
Cease | 03_trade_secret_misappropriation.pdf | 
Cease | 04_defamation_online_review.pdf | 
Cease | 05_patent_infringement_medical_device.pdf | 
Cease | 06_harassment_workplace.pdf | 
Cease | 07_software_license_violation.pdf | 
Cease | 08_non_compete_violation.pdf | 
Cease | 09_copyright_infringement_music.pdf | 
Cease | 10_breach_of_contract_nda.pdf | 
Cease | LOA2.pdf | 
Cease | LOA3.pdf | 
Cease | LOA4.pdf | 
Cease | LOA5.pdf | 
Cease | LOA6.pdf | 
Cease | LOA7.pdf | 
Cease | LOA8.pdf | 
Cease | LOA9.pdf | 
Cease | LoA1.pdf | 
Irrelevant | bw_doc_1.pdf | 
Irrelevant | bw_doc_2.pdf | 
Irrelevant | bw_doc_3.pdf | 
Irrelevant | bw_doc_4.pdf | 
Irrelevant | bw_doc_5.pdf | 
Cease | notice_1.pdf | 
Cease | notice_2.pdf | *Human Reviewed*
Cease | notice_3.pdf | 
Cease | notice_4.pdf | 
Cease | notice_5.pdf | *Human Reviewed*

TOTALS:
  Cease      : 24
  Irrelevant 

#END